# K6: TF-IDF, logistička regresija naspram stabla odlučivanja (online prodavnica odeće)
**Tip:** Blok 2, zadatak 2. Reprezentacija: TF-IDF. Dva klasifikatora koja se porede, pa podešavanje stabla.

### Zadatak
U fajlu `E-Commerce_Reviews_for_class.csv` su komentari kupaca online prodavnice odeće (`Clothing ID`, `Age`, `Title`, `Review Text`, `Rating`).
- Napraviti kolonu `review_content` spajanjem `Title` i `Review Text`.
- Od kolone `Rating` napraviti `sentiment` (1 i 2 negativan, 3 neutralan, 4 i 5 pozitivan).
- Transformisati `review_content` primenom **TF-IDF**. Proceniti da li još neku varijablu treba uključiti.
- Prvi model: **logistička regresija**, podrazumevani hiperparametri. Drugi model: **stablo odlučivanja**, podrazumevani hiperparametri. Uporediti ih.
- Treći model: **podešeno stablo** (jedan hiperparametar za TF-IDF, jedan za stablo, 5-fold kros-validacija), uz komentar šta bi sve podešavao.
- Uporediti sva tri modela i `DummyClassifier` kao donju granicu.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn import metrics
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # imblearn Pipeline radi i bez SMOTE-a, pa moze da ostane uvek

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

import matplotlib.pyplot as plt
import seaborn as sb

from pathlib import Path
from IPython.display import display

import spacy

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'

### Šta je urađeno
- Učitani podaci, napravljena `review_content` (prazan naslov se menja praznim stringom da ne "pokvari" spoj).
- Napravljena izlazna varijabla `sentiment` (pozitivnih je najviše).
- Pregledani `Age` (kdeplot) i `Clothing ID` (broj vrednosti).
- Odluka: ulaz je samo `review_content`.
- Podela 80:20 sa `stratify`.

In [ ]:
data = pd.read_csv(DATA_DIR / 'E-Commerce_Reviews_for_class.csv')
data.info()

In [ ]:
data['Title'] = data['Title'].fillna('')
data = data.dropna(subset=['Review Text']).reset_index(drop=True)
data['review_content'] = data['Title'] + '. ' + data['Review Text']
print(data['review_content'].head(3).tolist())

In [ ]:
def rating_to_sentiment(rating: int):
    if rating < 3:
        return 'negativan'
    return 'neutralan' if rating == 3 else 'pozitivan'

data['sentiment'] = data['Rating'].apply(rating_to_sentiment)
print(data['sentiment'].value_counts(normalize=True))

In [ ]:
plt.figure(figsize=(10, 6))
sb.kdeplot(data=data, x='Age', hue='sentiment',
           fill=True, common_norm=False, alpha=0.5, linewidth=2)
plt.title('Starost kupca po sentimentu')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print('Broj razlicitih proizvoda:', data['Clothing ID'].nunique())

In [ ]:
X = data['review_content']
y = data['sentiment']
y_encodings, y_levels = pd.factorize(y)
print(list(y_levels))

X_train, X_test, y_train, y_test = train_test_split(X, y_encodings,
                                                    test_size=0.2,
                                                    random_state=RAND_STATE,
                                                    stratify=y_encodings)
print(X_train.shape, X_test.shape)

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. spaCy `en_core_web_sm` i `spacy_srediTekst`
2. `TfidfVectorizer(...)`
3. Model 1: `Pipeline` sa `LogisticRegression(max_iter=2000)`. Model 2: isti vektorizer sa `DecisionTreeClassifier(random_state=RAND_STATE)`
4. Metrike za oba i tumačenje za jedan
5. Komentar o hiperparametrima stabla, `GridSearchCV` (npr. `min_df` + `max_depth`)
6. `DummyClassifier(strategy='most_frequent')` i tabela sa `get_model_metrics` za sva 4

---
## Dati kod
Ovo dobijaš uz zadatak. Pokreni ćeliju kad stigneš do poređenja modela.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): poredjenje modela
def get_model_metrics(y_true: np.ndarray, y_pred: np.ndarray, model_name: str):
    report = classification_report(y_true, y_pred, output_dict=True)
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    return {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred)
    }